**Вариант 10**

**Практическая работа 6. Подготовка фич и интеграция источников**

Цель работы: научиться объединять несколько источников данных (заказы, товары, события) и строить на их основе признаки (фичи) для последующей аналитики и простых ML-моделей.

Задание варианта 10:

1. собрать максимально полный `user_features` с агрегатами по заказам, событиям и категориям;
2. закодировать все категориальные признаки (one-hot или частотно);
3. сохранить итоговый датасет в формате Parquet и подготовить его для обучения модели.

Исходные данные

Используются файлы из папки `for_lab_3` — те же наборы, что и в практических работах 3–5:

| Файл | Формат | Содержимое |
|---|---|---|
| `orders_2024_01.csv`, `orders_2024_02.csv`, `orders_2024_03.csv` | CSV | заказы интернет-магазина по месяцам |
| `products.xlsx` | Excel | листы `products` (товары) и `categories` (категории) |
| `events_records.json` | JSON (records) | события/клики пользователей |
| `events_lines.json` | JSON Lines | события/клики пользователей |

Структура заказов: `order_id`, `user_id`, `order_datetime`, `product_id`, `price`, `quantity`, `status`.

Структура событий: `event_id`, `user_id`, `event_time`, `event_type`, `page`.

В заказах 15 000 позиций за январь–март 2024 года, в событиях — 4 000 записей, в справочнике — 30 товаров и 5 категорий. Итоговые файлы сохраняются в папку `for_lab_6`.

In [1]:
import glob
import json
import os

import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)

# папка с исходными файлами практической работы
DATA_DIR = 'for_lab_3'
if not os.path.isdir(DATA_DIR):
    DATA_DIR = 'data'

# папка для итоговых файлов
OUT_DIR = 'for_lab_6'
os.makedirs(OUT_DIR, exist_ok=True)

print('Папка с исходными данными:', os.path.abspath(DATA_DIR))
print('Папка для результатов:   ', os.path.abspath(OUT_DIR))
print()
for path in sorted(glob.glob(os.path.join(DATA_DIR, '*'))):
    print(f'{os.path.basename(path):<25}{os.path.getsize(path):>10} байт')

Папка с исходными данными: D:\учёба\vsuet\ims\for_lab_3
Папка для результатов:    D:\учёба\vsuet\ims\for_lab_6

events_lines.json            239555 байт
events_records.json          224211 байт
orders_2024_01.csv           244027 байт
orders_2024_02.csv           245248 байт
orders_2024_03.csv           250278 байт
products.xlsx                  6526 байт
user_profile.parquet          11919 байт


**1. Собрать максимально полный user_features с агрегатами по заказам, событиям и категориям**

In [2]:
orders_files = sorted(glob.glob(os.path.join(DATA_DIR, 'orders_2024_*.csv')))

orders_dtypes = {
    'order_id': 'int32',
    'user_id': 'int32',
    'product_id': 'int16',
    'quantity': 'int16',
    'price': 'float64',
    'status': 'category',
}

df_orders = pd.concat(
    [pd.read_csv(path, dtype=orders_dtypes, parse_dates=['order_datetime']) for path in orders_files],
    ignore_index=True,
)

print('Заказы:', df_orders.shape)
print(df_orders.dtypes.to_string())
print()
print(df_orders.head().to_string(index=False))
print()
print('Пропусков:               ', int(df_orders.isna().sum().sum()))
print('Уникальных пользователей:', df_orders['user_id'].nunique())
print('Уникальных заказов:      ', df_orders['order_id'].nunique())
print('Период:', df_orders['order_datetime'].min(), '—', df_orders['order_datetime'].max())

Заказы: (15000, 7)
order_id                   int32
user_id                    int32
order_datetime    datetime64[us]
product_id                 int16
price                    float64
quantity                   int16
status                  category

 order_id  user_id      order_datetime  product_id    price  quantity   status
        1     1044 2024-01-03 18:39:26          26 14038.15         1 refunded
        2     1079 2024-01-03 17:45:43          16  9172.60         1     paid
        3     1041 2024-01-06 22:46:38          25  9035.26         2     paid
        4     1083 2024-01-08 21:03:51           9 15234.33         3     paid
        5     1068 2024-01-03 23:26:53          24  4134.38         2     paid

Пропусков:                0
Уникальных пользователей: 100
Уникальных заказов:       15000
Период: 2024-01-01 00:10:25 — 2024-03-31 23:34:00


In [3]:
excel_path = os.path.join(DATA_DIR, 'products.xlsx')
sheets = pd.read_excel(excel_path, sheet_name=['products', 'categories'])
df_products = sheets['products']
df_categories = sheets['categories']

df_prod_full = (
    df_products
    .drop(columns=['category_name'])
    .merge(
        df_categories[['category_id', 'category_name']],
        on='category_id',
        how='left',
        validate='many_to_one',
    )
)

print('Справочник категорий:')
print(df_categories[['category_id', 'category_name']].to_string(index=False))
print()
print('Товары с категориями:')
print(df_prod_full.head().to_string(index=False))
print()
print('Товаров:', df_prod_full['product_id'].nunique(), '| категорий:', df_prod_full['category_name'].nunique())
print('Пропусков в справочнике:', int(df_prod_full.isna().sum().sum()))

Справочник категорий:
 category_id category_name
           1   Electronics
           2          Home
           3         Sport
           4       Fashion
           5          Toys

Товары с категориями:
 product_id product_name  category_id  weight_kg category_name
          1    Product 1            1      11.55   Electronics
          2    Product 2            3       2.22         Sport
          3    Product 3            5      11.43          Toys
          4    Product 4            1       4.17   Electronics
          5    Product 5            5       1.81          Toys

Товаров: 30 | категорий: 5
Пропусков в справочнике: 0


In [4]:
def read_events_records(path):
    '''Читает JSON в формате records (список объектов).'''
    with open(path, encoding='utf-8') as f:
        raw = json.load(f)

    df = pd.DataFrame(raw)

    # файл может быть сохранён транспонированным: 5 строк-признаков и 2000 столбцов
    if 'event_id' not in df.columns:
        df = df.T.reset_index(drop=True)
        df.columns = ['event_id', 'user_id', 'event_time', 'event_type', 'page']

    return df


df_events = pd.concat(
    [
        read_events_records(os.path.join(DATA_DIR, 'events_records.json'),),
        pd.read_json(os.path.join(DATA_DIR, 'events_lines.json'), lines=True),
    ],
    ignore_index=True,
)

for col in ['event_id', 'user_id']:
    df_events[col] = pd.to_numeric(df_events[col], errors='coerce').astype('int32')

df_events['event_time'] = pd.to_datetime(df_events['event_time'])
df_events['event_type'] = df_events['event_type'].astype('category')
df_events['page'] = df_events['page'].astype('category')

print('События:', df_events.shape)
print(df_events.dtypes.to_string())
print()
print(df_events.head().to_string(index=False))
print()
print('Типы событий:')
print(df_events['event_type'].value_counts().to_string())
print()
print('Уникальных пользователей в событиях:', df_events['user_id'].nunique())
print('Пропусков в событиях:', int(df_events.isna().sum().sum()))

События: (4000, 5)
event_id               int32
user_id                int32
event_time    datetime64[us]
event_type          category
page                category

 event_id  user_id          event_time  event_type        page
        1     1085 2024-01-30 20:37:32   page_view /product/26
        2     1006 2024-02-24 22:55:13 add_to_cart /product/28
        3     1024 2024-02-14 17:49:43       click /product/27
        4     1062 2024-02-16 11:21:32       click  /product/1
        5     1012 2024-02-08 05:25:21 add_to_cart           /

Типы событий:
event_type
click          819
purchase       807
page_view      804
scroll         788
add_to_cart    782

Уникальных пользователей в событиях: 100
Пропусков в событиях: 0


In [5]:
df_orders['revenue'] = df_orders['price'] * df_orders['quantity']

df_orders_prod = df_orders.merge(
    df_prod_full,
    on='product_id',
    how='left',
    validate='many_to_one',
)

print('Заказы с товарами и категориями:', df_orders_prod.shape)
print(
    df_orders_prod[
        ['order_id', 'user_id', 'product_name', 'category_name', 'price', 'quantity', 'revenue', 'status']
    ].head().to_string(index=False)
)
print()
print('Пропусков после объединения:', int(df_orders_prod.isna().sum().sum()))
print('Товаров без категории:      ', int(df_orders_prod['category_name'].isna().sum()))
print(f"Суммарная выручка: {df_orders_prod['revenue'].sum():,.2f} руб.")

Заказы с товарами и категориями: (15000, 12)
 order_id  user_id product_name category_name    price  quantity  revenue   status
        1     1044   Product 26         Sport 14038.15         1 14038.15 refunded
        2     1079   Product 16         Sport  9172.60         1  9172.60     paid
        3     1041   Product 25         Sport  9035.26         2 18070.52     paid
        4     1083    Product 9          Toys 15234.33         3 45702.99     paid
        5     1068   Product 24   Electronics  4134.38         2  8268.76     paid

Пропусков после объединения: 0
Товаров без категории:       0
Суммарная выручка: 384,543,651.79 руб.


In [6]:
category_summary = (
    df_orders_prod
    .groupby('category_name', observed=True)
    .agg(
        revenue=('revenue', 'sum'),
        orders=('order_id', 'nunique'),
        items=('quantity', 'sum'),
    )
    .sort_values('revenue', ascending=False)
)
category_summary['revenue_share'] = category_summary['revenue'] / category_summary['revenue'].sum()

print('Выручка по категориям:')
print(
    category_summary.to_string(
        formatters={'revenue': '{:,.0f}'.format, 'revenue_share': '{:.1%}'.format}
    )
)
print()
print('Статусы заказов:')
print(df_orders_prod['status'].value_counts(normalize=True).to_string())

Выручка по категориям:
                  revenue  orders  items revenue_share
category_name                                         
Toys          116,067,056    4544  11337         30.2%
Sport         100,340,693    3900   9680         26.1%
Electronics    77,420,436    3023   7637         20.1%
Fashion        51,968,337    2002   5048         13.5%
Home           38,747,129    1531   3863         10.1%

Статусы заказов:
status
paid         0.802133
cancelled    0.147467
refunded     0.050400


In [7]:
order_level = (
    df_orders_prod
    .groupby(['user_id', 'order_id'], observed=True)
    .agg(
        order_revenue=('revenue', 'sum'),
        order_items=('quantity', 'sum'),
        order_datetime=('order_datetime', 'min'),
        status=('status', 'first'),
    )
    .reset_index()
)

orders_by_user = (
    order_level
    .groupby('user_id', observed=True)
    .agg(
        orders_count=('order_id', 'nunique'),
        revenue_sum=('order_revenue', 'sum'),
        avg_check=('order_revenue', 'mean'),
        median_check=('order_revenue', 'median'),
        items_count=('order_items', 'sum'),
        avg_items_per_order=('order_items', 'mean'),
        paid_share=('status', lambda s: (s == 'paid').mean()),
        cancelled_share=('status', lambda s: (s == 'cancelled').mean()),
        refunded_share=('status', lambda s: (s == 'refunded').mean()),
        first_order=('order_datetime', 'min'),
        last_order=('order_datetime', 'max'),
    )
)

assortment = (
    df_orders_prod
    .groupby('user_id', observed=True)
    .agg(
        unique_products=('product_id', 'nunique'),
        unique_categories=('category_name', 'nunique'),
    )
)

orders_by_user = orders_by_user.join(assortment)

print('Агрегаты по заказам:', orders_by_user.shape)
print(orders_by_user.head().to_string())

Агрегаты по заказам: (100, 13)
         orders_count  revenue_sum     avg_check  median_check  items_count  avg_items_per_order  paid_share  cancelled_share  refunded_share         first_order          last_order  unique_products  unique_categories
user_id                                                                                                                                                                                                                  
1001              177   4435767.09  25060.831017      19560.72          454             2.564972    0.802260         0.146893        0.050847 2024-01-02 02:51:37 2024-03-31 21:32:19               30                  5
1002              123   2830004.52  23008.166829      18106.91          281             2.284553    0.821138         0.130081        0.048780 2024-01-02 12:57:34 2024-03-31 19:45:16               29                  5
1003              156   4025266.18  25802.988333      19919.98          389             2.493590 

In [8]:
events_by_user = (
    df_events
    .groupby('user_id', observed=True)
    .agg(
        events_count=('event_id', 'count'),
        page_view=('event_type', lambda s: (s == 'page_view').sum()),
        click=('event_type', lambda s: (s == 'click').sum()),
        scroll=('event_type', lambda s: (s == 'scroll').sum()),
        add_to_cart=('event_type', lambda s: (s == 'add_to_cart').sum()),
        purchase=('event_type', lambda s: (s == 'purchase').sum()),
        unique_pages=('page', 'nunique'),
        first_event=('event_time', 'min'),
        last_event=('event_time', 'max'),
    )
)

print('Агрегаты по событиям:', events_by_user.shape)
print(events_by_user.head().to_string())
print()
print('Сумма событий по типам:')
print(events_by_user[['page_view', 'click', 'scroll', 'add_to_cart', 'purchase']].sum().to_string())

Агрегаты по событиям: (100, 9)
         events_count  page_view  click  scroll  add_to_cart  purchase  unique_pages         first_event          last_event
user_id                                                                                                                     
1001               54         13     12       6            7        16            22 2024-01-01 06:25:31 2024-03-31 22:29:02
1002               41          8      7       7            7        12            22 2024-01-04 12:58:00 2024-03-31 21:24:08
1003               39          3      6       8            6        16            20 2024-01-09 20:43:11 2024-03-29 20:53:24
1004               37          4      3       7            8        15            22 2024-01-04 14:36:56 2024-03-27 04:58:11
1005               44          6      6       9           10        13            25 2024-01-01 16:44:30 2024-03-30 04:58:21

Сумма событий по типам:
page_view      804
click          819
scroll         788
add_to_cart 

In [9]:
user_cat = (
    df_orders_prod
    .groupby(['user_id', 'category_name'], observed=True)
    .agg(
        cat_revenue=('revenue', 'sum'),
        cat_items=('quantity', 'sum'),
    )
    .reset_index()
)

cat_revenue = (
    user_cat
    .pivot_table(
        index='user_id',
        columns='category_name',
        values='cat_revenue',
        aggfunc='sum',
        fill_value=0,
    )
)
cat_revenue.columns = [f'cat_revenue_{str(c).lower()}' for c in cat_revenue.columns]

cat_share = cat_revenue.div(cat_revenue.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
cat_share.columns = [c.replace('cat_revenue_', 'cat_share_') for c in cat_revenue.columns]

top_category = cat_revenue.idxmax(axis=1).str.replace('cat_revenue_', '', regex=False)
top_category_share = cat_share.max(axis=1)
category_diversity = (cat_revenue > 0).sum(axis=1)

print('Выручка по категориям (фрагмент):')
print(cat_revenue.head().round(0).to_string())
print()
print('Самые доходные категории пользователей:')
print(top_category.value_counts().to_string())
print()
print('Доля топ-категории в выручке:')
print(top_category_share.describe().round(3).to_string())

Выручка по категориям (фрагмент):
         cat_revenue_electronics  cat_revenue_fashion  cat_revenue_home  cat_revenue_sport  cat_revenue_toys
user_id                                                                                                     
1001                   1079367.0             394191.0          510953.0          1212973.0         1238283.0
1002                    430626.0             383410.0          409831.0           703656.0          902482.0
1003                    797357.0             526169.0          421733.0           831484.0         1448524.0
1004                    989113.0             506033.0          296547.0          1111600.0          884738.0
1005                    437808.0             615829.0          290514.0           820041.0         1021461.0

Самые доходные категории пользователей:
toys           67
sport          30
electronics     3

Доля топ-категории в выручке:
count    100.000
mean       0.317
std        0.040
min        0.246
25%      

In [10]:
df_events_enriched = df_events.copy()

df_events_enriched['product_id'] = (
    df_events_enriched['page']
    .astype('string')
    .str.extract(r'/product/(\d+)', expand=False)
    .astype('Int32')
)

df_events_enriched = df_events_enriched.merge(
    df_prod_full[['product_id', 'category_name']],
    on='product_id',
    how='left',
    validate='many_to_one',
)

page = df_events_enriched['page'].astype('string')
df_events_enriched['page_type'] = np.select(
    [
        page.str.startswith('/product/', na=False),
        page.eq('/'),
        page.eq('/cart'),
    ],
    ['product', 'home', 'cart'],
    default='other',
)

print('События с извлечённым product_id:', int(df_events_enriched['product_id'].notna().sum()))
print('Доля событий на страницах товаров:',
      f'{df_events_enriched["page_type"].eq("product").mean():.1%}')
print()
print('Типы страниц:')
print(pd.Series(df_events_enriched['page_type']).value_counts().to_string())

# события на страницах товаров с известной категорией
product_events = df_events_enriched.dropna(subset=['product_id', 'category_name'])

event_cat_counts = (
    product_events
    .groupby(['user_id', 'category_name'], observed=True)
    .size()
    .unstack(fill_value=0)
)
event_cat_counts.columns = [f'events_cat_{str(c).lower()}' for c in event_cat_counts.columns]

event_cat_share = event_cat_counts.div(
    df_events.groupby('user_id', observed=True)['event_id'].count(),
    axis=0,
).fillna(0)
event_cat_share.columns = [c.replace('events_cat_', 'events_share_') for c in event_cat_counts.columns]

top_event_category = event_cat_counts.idxmax(axis=1).str.replace('events_cat_', '', regex=False)

page_type_counts = (
    df_events_enriched
    .groupby(['user_id', 'page_type'], observed=True)
    .size()
    .unstack(fill_value=0)
)
page_type_counts.columns = [f'events_page_{str(c).lower()}' for c in page_type_counts.columns]
top_page_type = page_type_counts.idxmax(axis=1).str.replace('events_page_', '', regex=False)

print()
print('События по категориям (фрагмент):')
print(event_cat_counts.head().to_string())
print()
print('Топ-категории по событиям:')
print(top_event_category.value_counts().to_string())

События с извлечённым product_id: 2832
Доля событий на страницах товаров: 70.8%

Типы страниц:
page_type
product    2832
cart        586
home        582

События по категориям (фрагмент):
         events_cat_electronics  events_cat_fashion  events_cat_home  events_cat_sport  events_cat_toys
user_id                                                                                                
1001                          7                   5                2                11               13
1002                          6                   2                4                10               10
1003                          2                   3                3                 7               11
1004                          9                   6                1                 6                7
1005                          7                   4                4                 6               11

Топ-категории по событиям:
toys           53
sport          29
electronics    14
fa

In [11]:
user_features = (
    orders_by_user
    .join(events_by_user, how='outer')
    .join(cat_revenue, how='outer')
    .join(cat_share, how='outer')
    .join(event_cat_counts, how='outer')
    .join(event_cat_share, how='outer')
    .join(page_type_counts, how='outer')
)

user_features['top_category'] = top_category
user_features['top_category_share'] = top_category_share
user_features['category_diversity'] = category_diversity
user_features['top_event_category'] = top_event_category
user_features['top_page_type'] = top_page_type

# счётчики и доли: пропуски означают отсутствие активности
numeric_cols = user_features.select_dtypes(include='number').columns
user_features[numeric_cols] = user_features[numeric_cols].fillna(0)

for col in ['top_category', 'top_event_category', 'top_page_type']:
    user_features[col] = user_features[col].fillna('no_activity')

# общий период активности по двум источникам
user_features['first_activity'] = user_features[['first_order', 'first_event']].min(axis=1)
user_features['last_activity'] = user_features[['last_order', 'last_event']].max(axis=1)
user_features['lifetime_days'] = (
    user_features['last_activity'] - user_features['first_activity']
).dt.days

snapshot = user_features['last_activity'].max() + pd.Timedelta(days=1)
user_features['recency_days'] = (snapshot - user_features['last_activity']).dt.days

# плотность событий и конверсия (защита от деления на ноль)
user_features['events_per_order'] = (
    user_features['events_count'] / user_features['orders_count'].replace(0, np.nan)
)
user_features['purchase_rate'] = (
    user_features['purchase'] / user_features['events_count'].replace(0, np.nan)
)
user_features['has_orders'] = (user_features['orders_count'] > 0).astype('int8')
user_features['has_events'] = (user_features['events_count'] > 0).astype('int8')

user_features[['events_per_order', 'purchase_rate']] = (
    user_features[['events_per_order', 'purchase_rate']].fillna(0)
)

print('Полный user_features:', user_features.shape)
print('Пользователей:', user_features.shape[0])
print()
print(
    user_features[
        ['orders_count', 'revenue_sum', 'avg_check', 'events_count', 'purchase',
         'top_category', 'lifetime_days', 'recency_days']
    ].head(10).round(2).to_string()
)

Полный user_features: (100, 58)
Пользователей: 100

         orders_count  revenue_sum  avg_check  events_count  purchase top_category  lifetime_days  recency_days
user_id                                                                                                        
1001              177   4435767.09   25060.83            54        16         toys             90             1
1002              123   2830004.52   23008.17            41        12         toys             89             1
1003              156   4025266.18   25802.99            39        16         toys             89             1
1004              147   3788030.56   25768.92            37        15        sport             90             1
1005              141   3185653.37   22593.29            44        13         toys             90             1
1006              161   4160640.89   25842.49            36         5         toys             89             2
1007              157   4064170.75   25886.44       

**2. Закодировать все категориальные признаки (one-hot или частотно)**

In [12]:
# --- частотное кодирование статусов и типов событий ---
status_freq = df_orders['status'].astype('string').value_counts(normalize=True)
df_orders['status_freq'] = (
    df_orders['status'].astype('string').map(status_freq).astype('float32')
)

event_type_freq = df_events['event_type'].astype('string').value_counts(normalize=True)
df_events['event_type_freq'] = (
    df_events['event_type'].astype('string').map(event_type_freq).astype('float32')
)

freq_by_user = (
    df_orders
    .groupby('user_id', observed=True)
    .agg(
        status_freq_mean=('status_freq', 'mean'),
        status_freq_std=('status_freq', 'std'),
    )
    .join(
        df_events
        .groupby('user_id', observed=True)
        .agg(
            event_type_freq_mean=('event_type_freq', 'mean'),
            event_type_freq_std=('event_type_freq', 'std'),
        ),
        how='outer',
    )
)

user_features = user_features.join(freq_by_user)
user_features[['status_freq_mean', 'status_freq_std',
               'event_type_freq_mean', 'event_type_freq_std']] = (
    user_features[['status_freq_mean', 'status_freq_std',
                   'event_type_freq_mean', 'event_type_freq_std']].fillna(0)
)

print('Частоты статусов:')
print(status_freq.round(4).to_string())
print()
print('Частоты типов событий:')
print(event_type_freq.round(4).to_string())
print()
print('Признаки частотного кодирования по пользователям:')
print(user_features[['status_freq_mean', 'event_type_freq_mean']].head().round(4).to_string())

Частоты статусов:
status
paid         0.8021
cancelled    0.1475
refunded     0.0504

Частоты типов событий:
event_type
click          0.2047
purchase       0.2018
page_view       0.201
scroll          0.197
add_to_cart    0.1955

Признаки частотного кодирования по пользователям:
         status_freq_mean  event_type_freq_mean
user_id                                        
1001               0.6677                0.2009
1002               0.6803                0.2002
1003               0.7031                0.2002
1004               0.7009                0.1997
1005               0.6793                0.1997


In [13]:
# --- one-hot кодирование пользовательских категорий ---
onehot_cols = ['top_category', 'top_event_category', 'top_page_type']

onehot = pd.get_dummies(
    user_features[onehot_cols].astype('string'),
    prefix=onehot_cols,
    dtype='int8',
)

print('One-hot столбцы:', onehot.shape[1])
print(list(onehot.columns))
print()
print('Проверка: в каждой строке ровно одна единица по каждому блоку:')
for col in onehot_cols:
    block = onehot.filter(like=f'{col}_')
    print(f'  {col}: {int((block.sum(axis=1) == 1).sum())} из {len(block)}')

One-hot столбцы: 9
['top_category_electronics', 'top_category_sport', 'top_category_toys', 'top_event_category_electronics', 'top_event_category_fashion', 'top_event_category_home', 'top_event_category_sport', 'top_event_category_toys', 'top_page_type_product']

Проверка: в каждой строке ровно одна единица по каждому блоку:
  top_category: 100 из 100
  top_event_category: 100 из 100
  top_page_type: 100 из 100


In [14]:
service_cols = [
    'first_order', 'last_order', 'first_event', 'last_event',
    'first_activity', 'last_activity',
]

drop_for_model = service_cols + onehot_cols
feature_cols = [c for c in user_features.columns if c not in drop_for_model]

X = pd.concat([user_features[feature_cols], onehot], axis=1)

# константные столбцы бесполезны для модели
constant_cols = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]
X = X.drop(columns=constant_cols)

print('Удалены константные признаки:', constant_cols if constant_cols else 'нет')
print()
print('Матрица признаков:', X.shape)
print('Столбцов до кодирования категорий:', len(feature_cols))
print('One-hot столбцов:', onehot.shape[1])
print()
print('Нечисловых столбцов:', X.select_dtypes(exclude='number').shape[1])
print('Пропусков в матрице: ', int(X.isna().sum().sum()))
print()
print('Типы данных:')
print(X.dtypes.value_counts().to_string())

Удалены константные признаки: ['unique_categories', 'category_diversity', 'has_orders', 'has_events', 'top_page_type_product']

Матрица признаков: (100, 57)
Столбцов до кодирования категорий: 53
One-hot столбцов: 9

Нечисловых столбцов: 0
Пропусков в матрице:  0

Типы данных:
float64    25
int64      19
int8        8
float32     4
int16       1


**3. Сохранить итоговый датасет в формате Parquet и подготовить его для обучения модели**

In [15]:
repeat_share = float((user_features['orders_count'] > 1).mean())
print(f'Доля пользователей с повторным заказом (orders_count > 1): {repeat_share:.1%}')
print('Такой таргет был бы константным, поэтому используем другую постановку.')
print()

# случайное разбиение пользователей 80/20
rng = np.random.default_rng(42)
user_ids = user_features.index.to_numpy().copy()
rng.shuffle(user_ids)

n_test = int(round(len(user_ids) * 0.2))
test_idx = np.sort(user_ids[:n_test])
train_idx = np.sort(user_ids[n_test:])

# порог высокодоходности — медиана только по train
revenue_threshold = float(user_features.loc[train_idx, 'revenue_sum'].median())
target = (
    user_features['revenue_sum'] > revenue_threshold
).astype('int8').rename('is_high_value')

# revenue_sum напрямую определяет таргет — исключаем из признаков
X_model = X.drop(columns=['revenue_sum']).sort_index()
y = target.sort_index()

X_train, X_test = X_model.loc[train_idx], X_model.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

print(f'Порог высокодоходности (медиана train): {revenue_threshold:,.2f} руб.')
print('Матрица для модели:', X_model.shape)
print()
print('Таргет is_high_value:')
print(y.value_counts().to_string())
print(f'Доля высокодоходных пользователей: {y.mean():.1%}')
print()
print('Разбиение:')
print(f'  train: {X_train.shape[0]} пользователей, {X_train.shape[1]} признаков')
print(f'  test:  {X_test.shape[0]} пользователей, {X_test.shape[1]} признаков')
print(f'  доля высокодоходных в train: {y_train.mean():.1%}')
print(f'  доля высокодоходных в test:  {y_test.mean():.1%}')

Доля пользователей с повторным заказом (orders_count > 1): 100.0%
Такой таргет был бы константным, поэтому используем другую постановку.

Порог высокодоходности (медиана train): 3,956,270.92 руб.
Матрица для модели: (100, 56)

Таргет is_high_value:
is_high_value
0    51
1    49
Доля высокодоходных пользователей: 49.0%

Разбиение:
  train: 80 пользователей, 56 признаков
  test:  20 пользователей, 56 признаков
  доля высокодоходных в train: 50.0%
  доля высокодоходных в test:  45.0%


In [16]:
majority_class = int(round(y_train.mean()))
baseline_const = float((y_test == majority_class).mean())

check_threshold = float(X_train['avg_check'].median())
rule_pred = (X_test['avg_check'] > check_threshold).astype('int8')
baseline_rule = float((y_test == rule_pred).mean())

print('Baseline-проверки на тестовой выборке:')
print(f'  константный прогноз:               {baseline_const:.3f}')
print(f'  правило по avg_check (медиана):    {baseline_rule:.3f}')
print()
print('Топ-10 признаков по корреляции с таргетом (по модулю):')
corr = (
    X_model.assign(is_high_value=y)
    .corr(numeric_only=True)['is_high_value']
    .drop('is_high_value')
    .abs()
    .sort_values(ascending=False)
)
print(corr.head(10).round(3).to_string())

Baseline-проверки на тестовой выборке:
  константный прогноз:               0.550
  правило по avg_check (медиана):    0.600

Топ-10 признаков по корреляции с таргетом (по модулю):
items_count                0.734
orders_count               0.690
avg_check                  0.507
median_check               0.457
cat_revenue_fashion        0.434
cat_revenue_toys           0.423
cat_revenue_sport          0.371
cat_revenue_electronics    0.365
cat_revenue_home           0.363
avg_items_per_order        0.285


In [17]:
full_out = user_features.reset_index()

model_out = X_model.copy()
model_out.insert(0, 'user_id', model_out.index)
model_out['is_high_value'] = y.to_numpy()

train_out = X_train.copy()
train_out.insert(0, 'user_id', train_out.index)
train_out['is_high_value'] = y_train.to_numpy()

test_out = X_test.copy()
test_out.insert(0, 'user_id', test_out.index)
test_out['is_high_value'] = y_test.to_numpy()

paths = {
    'user_features_full.parquet': (full_out, 'parquet'),
    'user_features_full.csv': (full_out, 'csv'),
    'user_features_model.parquet': (model_out, 'parquet'),
    'train.parquet': (train_out, 'parquet'),
    'test.parquet': (test_out, 'parquet'),
}

for name, (frame, fmt) in paths.items():
    path = os.path.join(OUT_DIR, name)
    if fmt == 'parquet':
        frame.to_parquet(path, index=False)
    else:
        frame.to_csv(path, index=False)
    print(f'{name:<32}{os.path.getsize(path):>10} байт   {frame.shape}')

print()
print('Полный набор:      ', full_out.shape)
print('Матрица для модели:', model_out.shape)
print('Train / test:      ', train_out.shape, '/', test_out.shape)

user_features_full.parquet           69217 байт   (100, 63)
user_features_full.csv               67039 байт   (100, 63)
user_features_model.parquet          60760 байт   (100, 58)
train.parquet                        56046 байт   (80, 58)
test.parquet                         42047 байт   (20, 58)

Полный набор:       (100, 63)
Матрица для модели: (100, 58)
Train / test:       (80, 58) / (20, 58)


In [18]:
size_full_csv = os.path.getsize(os.path.join(OUT_DIR, 'user_features_full.csv'))
size_full_parquet = os.path.getsize(os.path.join(OUT_DIR, 'user_features_full.parquet'))

size_table = pd.DataFrame(
    [
        {
            'Формат': 'Parquet',
            'Файл': 'user_features_full.parquet',
            'Размер, КБ': size_full_parquet / 1024,
            'Типы сохраняются': 'да',
        },
        {
            'Формат': 'CSV',
            'Файл': 'user_features_full.csv',
            'Размер, КБ': size_full_csv / 1024,
            'Типы сохраняются': 'нет',
        },
    ]
)

print('Сравнение форматов:')
print(size_table.to_string(index=False, formatters={'Размер, КБ': '{:.1f}'.format}))
print()
print(f'Отношение размеров Parquet / CSV: {size_full_parquet / size_full_csv:.2f}')
print('На 100 строках разница почти незаметна: служебные метаданные Parquet')
print('сопоставимы с самими данными. Выигрыш формата — в сохранении типов')
print('и в сжатии при росте числа строк и столбцов.')
print()

check = pd.read_parquet(os.path.join(OUT_DIR, 'user_features_full.parquet'))
print('Проверка чтения Parquet обратно:', check.shape)
print(check[['user_id', 'orders_count', 'revenue_sum', 'events_count',
             'top_category', 'lifetime_days']].head().to_string(index=False))

Сравнение форматов:
 Формат                       Файл Размер, КБ Типы сохраняются
Parquet user_features_full.parquet       67.6               да
    CSV     user_features_full.csv       65.5              нет

Отношение размеров Parquet / CSV: 1.03
На 100 строках разница почти незаметна: служебные метаданные Parquet
сопоставимы с самими данными. Выигрыш формата — в сохранении типов
и в сжатии при росте числа строк и столбцов.



Проверка чтения Parquet обратно: (100, 63)
 user_id  orders_count  revenue_sum  events_count top_category  lifetime_days
    1001           177   4435767.09            54         toys             90
    1002           123   2830004.52            41         toys             89
    1003           156   4025266.18            39         toys             89
    1004           147   3788030.56            37        sport             90
    1005           141   3185653.37            44         toys             90
